# Lab 03B: A CrewAI Research Crew — Context Handoff + Memory

**Week 3 — Agentic AI: Building Autonomous Intelligent Systems**

In Lab 03A you built reflection and memory *by hand*. Now you'll use a real framework, **CrewAI**, where **agents**, **tasks**, and **memory** are first-class building blocks. You'll build a three-agent research crew — a **researcher**, an **analyst**, and a **writer** — where each agent's output becomes the next agent's **context**. Then you'll turn on CrewAI's built-in **memory** and see how it lets the crew carry context *across runs*, not just within one.

## Introduction

This lesson answers:

- What is a CrewAI crew (Agents, Tasks, Crew, Process)?
- How does one agent hand its work off to the next?
- What is the difference between **task-local state** (context within a run) and **remembered context** (memory across runs)?
- What does a framework give you that hand-writing the loop does not?

## Learning Goals

After completing this lesson you will be able to:

- Define **Agents** (role / goal / backstory) and **Tasks** (description / expected_output).
- Chain tasks so each one's output becomes the next task's **`context`**.
- Run a sequential **Crew** and read each task's output.
- Enable CrewAI **memory** and explain task-local state vs. remembered context across runs.

This lab is on the **API-key** track and talks to Gemini through CrewAI + LiteLLM.

## What is a CrewAI crew?

CrewAI models a workflow as a small team. You define **Agents** (each a persona with a `role`, `goal`, and `backstory`), give them **Tasks** (each a `description` + `expected_output`), and assemble them into a **Crew** that runs the tasks in order (`Process.sequential`). The magic is **`context`**: you list which earlier tasks a task depends on, and CrewAI feeds those outputs in automatically — that is the agent handoff.

```
   topic
     │
     ▼
   ┌──────────────┐  research brief
   │  researcher  │──────────────┐
   └──────────────┘              │  context
                                 ▼
                          ┌──────────────┐  analysis
                          │   analyst    │──────────────┐
                          └──────────────┘              │  context
                                                        ▼
                                                 ┌──────────────┐  final report
                                                 │    writer    │────────────▶
                                                 └──────────────┘
   each task's output becomes the next task's `context`.
   turn on Crew memory and that context can also persist ACROSS runs.
```

## Use cases

A sequential crew fits any task with clear, ordered phases:

- **Research -> analysis -> writing** (this lab): gather, interpret, present.
- **Draft -> review -> revise**: a writer, a critic, an editor.
- **Plan -> build -> test**: a planner, a coder, a tester.
- **Extract -> normalize -> report**: a data pipeline of specialist steps.

If steps are *independent* rather than ordered, you would fan them out in parallel instead (Week 2); a crew is for handoffs.

## Building blocks

- **Agent** — a worker defined by `role`, `goal`, and `backstory` (always a senior/expert persona here).
- **Task** — a `description` (what to do), an `expected_output` (the shape), and the `agent` that owns it.
- **`context`** — the list of earlier tasks whose outputs feed this one (the handoff).
- **Crew + Process** — the agents + tasks + an execution order (`sequential`).
- **Memory (optional)** — CrewAI's built-in short-term / long-term / entity memory, which carries context across runs.
- **An embedder** — memory stores and retrieves by similarity, so it needs an embedding model (we point it at Gemini).

## Considerations for trustworthy crews

- **Bound the crew.** More agents and hops mean more cost and more places to go wrong; add only the steps the task needs.
- **Context is not free.** Every handoff injects the prior output into the next prompt — long chains grow the context fast.
- **Memory is powerful and sticky.** Remembered context helps continuity but can also carry stale or wrong facts forward; know what is being persisted.
- **Constrain outputs where it matters.** Use `expected_output` (and schemas) so a downstream agent receives a predictable shape.
- **Keep handoffs inspectable.** Read each `task.output` so you can see exactly what each agent contributed.

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

The next cell installs CrewAI (which pulls in LiteLLM, the layer that lets CrewAI call Gemini). This is a larger install — give it a minute.

In [ ]:
!pip install -q crewai

> **Heads-up on the pip output:** you may see an `ERROR: pip's dependency resolver ...` line mentioning `bigframes` and `rich`. This is **expected in Colab and safe to ignore** — CrewAI upgrades `rich`, and Colab's preinstalled `bigframes` pins an older `rich`; this lab never uses `bigframes`, and the install still succeeded. If Colab shows a **"Restart session"** prompt after installing, click it (or **Runtime -> Restart session**) and re-run from the setup cell.

In [ ]:
import os

from google.colab import userdata
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")

# LiteLLM (under CrewAI) reads the key from this env var for gemini/* models.
os.environ["GEMINI_API_KEY"] = api_key

# The `gemini/` prefix routes to the Gemini API. NOTE: this lab uses gemini-2.5-flash, not the
# gemini-2.5-flash-lite the other labs use. A crew fires many larger, multi-step requests, and the
# lite model -- the cheapest and most heavily used -- is often overloaded (HTTP 503) for requests
# that size, while flash has the capacity to serve them reliably. num_retries adds a per-call retry
# as an extra cushion for the occasional transient blip.
llm = LLM(
    model="gemini/gemini-3.5-flash",
    api_key=api_key,
    temperature=0.3,
    num_retries=5,
)

# While CrewAI/LiteLLM is quietly retrying a 503, it still logs alarming red "ERROR" lines.
# Quiet those so the notebook output stays readable; our own retry messages (plain prints) still show.
import logging
for _noisy in ("crewai.flow.runtime", "LiteLLM", "litellm", "root"):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)

# ChromaDB's Google embedder (used by CrewAI memory) still imports the legacy google.generativeai
# package and prints a deprecation notice. It works fine; silence the notice to keep output clean.
import warnings
warnings.filterwarnings("ignore", message=r".*google\.generativeai.*")

# CrewAI can upload run "traces" to its hosted dashboard. We keep it off so nothing leaves this
# notebook and CrewAI stops printing its "Tracing Preference Saved" panel. (See the note by the run
# cell -- the dashboard is genuinely useful for your own later projects; flip this to "true" to try it.)
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Quick connectivity check. Gemini can return a transient 503 ("high demand") on any call,
# so we retry a few times, and if it still will not respond we warn instead of crashing.
import time

def _health_check(prompt, tries=4, wait=8):
    for attempt in range(1, tries + 1):
        try:
            return llm.call(prompt)
        except Exception as e:
            if "503" in str(e) and attempt < tries:
                print(f"  Model overloaded (503). Waiting {wait}s, then retrying ({attempt}/{tries})...")
                time.sleep(wait)
            else:
                raise

try:
    print(_health_check("Say 'Setup complete!' and nothing else."))
except Exception as e:
    print("Connectivity check could not complete (model may be busy):", str(e)[:120])
    print("That is OK -- num_retries retries a transient 503 when you run the crew below.")

## The three agents

Each agent is a senior persona. They share the same model; what makes one a researcher and another a writer is the `role` / `goal` / `backstory`.

In [ ]:
TOPIC = "the impact of AI agents on software development workflows"


researcher = Agent(
    role="Senior Research Analyst",
    goal="Gather a faithful, specific research brief on the topic.",
    backstory="You dig up the key facts, real challenges, and concrete examples; you cite concepts, not hype.",
    llm=llm,
    verbose=False,
)

analyst = Agent(
    role="Senior Strategy Analyst",
    goal="Turn research into ranked implications and clear recommendations.",
    backstory="You are opinionated and evidence-driven; you separate what matters from what does not.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Senior Technical Writer",
    goal="Turn analysis into a polished, readable short report.",
    backstory="You write tight, engaging prose for a general audience; no jargon dumps.",
    llm=llm,
    verbose=False,
)

## The three tasks -- and the handoff via `context`

Each task's `description` follows the Role / Context / Task / Constraints / Format shape. The part that makes this a *crew* and not three separate calls is the **`context=[...]`** argument on each `Task` -- that list **is** the handoff.

When you write `context=[research_task]` on the analyst's task, CrewAI takes whatever `research_task` produced and **injects it into the analyst's prompt automatically**. You never copy the brief across by hand; you just name the upstream task. The writer names both earlier tasks, so it receives both outputs.

Reading the `context=` lines top to bottom is the wiring diagram of the crew:

```
research_task    context=[]                          -> runs first; no upstream input
                       |
                       |  its output is injected as context
                       v
analysis_task    context=[research_task]             -> sees the researcher's brief
                       |
                       |  both outputs injected
                       v
writing_task     context=[research_task,             -> sees BOTH the brief
                          analysis_task]                 and the analysis
```

So the handoff is declarative: instead of passing arguments between function calls, each task *declares which earlier tasks it depends on*, and CrewAI threads the outputs through. The topic is embedded directly in the first task, so we call `kickoff()` with no templated inputs.

In [ ]:
research_task = Task(
    description=f"""# Context
You are the first step; the analyst and writer build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{TOPIC}""",
    expected_output="A structured research brief with clear section headers.",
    agent=researcher,
)

analysis_task = Task(
    description="""# Context
The researcher's brief is available to you as context.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
    expected_output="A ranked analysis with a summary, implications, and recommendations.",
    agent=analyst,
    context=[research_task],
)

writing_task = Task(
    description="""# Context
The research brief AND the analysis are available to you as context.

# Task
Write a polished, publication-ready short report that combines them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words.",
    agent=writer,
    context=[research_task, analysis_task],
)

## A note on 503 "model overloaded" errors

Gemini can return a transient **503 ("high demand")** when a model is busy. Two choices keep this lab robust:

1. **Model choice:** this lab uses `gemini-2.5-flash` (see setup). A crew fires many larger requests, and the cheaper `gemini-2.5-flash-lite` is frequently overloaded for that load, while `flash` has the capacity to serve it.
2. **Per-call retry:** `num_retries=5` on the `LLM` retries an individual call on a transient blip before it can fail the crew.

If Gemini has a broad capacity event, calls can still fail -- that is a server-side outage, not a bug. Wait a few minutes and run the cell again.

## Run the crew

`Process.sequential` runs the tasks in order and threads the `context` through. After the run, each task's result is on `task.output` — read them to see exactly what each agent handed off.

> **Async note (Colab):** Colab already runs an `asyncio` event loop, and this version of CrewAI refuses a *synchronous* `crew.kickoff()` from inside a running loop. So we use the async entry point **`await crew.kickoff_async()`** with top-level `await` — exactly the pattern from the async lab. (In a plain `.py` script with no running loop, `crew.kickoff()` works directly.)

> **Aside — CrewAI's tracing dashboard:** CrewAI can upload a step-by-step trace of each run (every agent, the exact prompts and responses, token counts, latency) to a hosted web dashboard for debugging. We keep it **off** in this lab (`CREWAI_TRACING_ENABLED=false` in setup) so nothing leaves your notebook and the output stays clean — everything you need is printed inline below. For your own real projects it is worth a look: set `tracing=True` on the `Crew` (it needs a free CrewAI account) to inspect runs in the UI.

In [ ]:
crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    verbose=False,
)

try:
    await crew.kickoff_async()

    print("=== RESEARCH BRIEF (researcher) ===")
    print(research_task.output.raw)
    print("\n=== ANALYSIS (analyst -- read the brief via context) ===")
    print(analysis_task.output.raw)
    print("\n=== FINAL REPORT (writer -- read both via context) ===")
    print(writing_task.output.raw)
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s) and the crew could not finish.")
        print("This is a transient, server-side capacity issue -- not a bug in your code or this lab.")
        print("Wait a few minutes and re-run this cell; single calls usually recover quickly.")
    else:
        raise

## What just happened: context handoff

You never passed the research text to the analyst by hand. Because `analysis_task` declared `context=[research_task]`, CrewAI injected the researcher's output into the analyst's prompt automatically; the writer got both. That is the crew handoff — coordination through `context` instead of manual argument passing.

But notice: that context is **task-local**. It lives only for this one `kickoff()`. Run the crew again and it starts fresh with no memory of the last run. Enabling **memory** changes that.

## Optional: enable CrewAI memory (context that survives across runs)

Set `memory=True` on the Crew and CrewAI keeps short-term, long-term, and entity memory *across* `kickoff()` calls — so a later run can recall what earlier runs established. Because memory retrieves by similarity, it needs an **embedder**; we point it at Gemini's embedding model so the lab stays Gemini-only (the default embedder is OpenAI).

The contrast to feel:
- **Without memory** (above): every run is isolated — only task-local context within the run.
- **With memory** (below): the crew accumulates a memory store that later runs read from.

> **Heads-up:** memory + the embedder config can vary by CrewAI version and needs a live Colab run to confirm. If the embedder line errors, check the CrewAI memory docs for the provider/model names your installed version expects.

> As above, we use `await memory_crew.kickoff_async()` because Colab has a running event loop.

In [ ]:
memory_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, writing_task],
    process=Process.sequential,
    memory=True,  # <-- turn on short-term / long-term / entity memory across runs
    embedder={
        # CrewAI renamed the Gemini embeddings provider: use "google-generativeai" (the Gemini API
        # path) -- "google-vertex" is the separate Vertex AI path, and the old bare "google" is gone.
        "provider": "google-generativeai",
        "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
    },
    verbose=False,
)

# Run 1 seeds the memory; Run 2 can recall it. (Same crew, run twice.)
try:
    print("--- Run 1 (seeds memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")

    print("\n--- Run 2 (can recall Run 1 from memory) ---")
    await memory_crew.kickoff_async()
    print(writing_task.output.raw[:300], "...")
except Exception as e:
    if "503" in str(e):
        print("Gemini is under heavy load right now (repeated 503s); the memory demo could not finish.")
        print("This is a transient server-side issue -- wait a few minutes and re-run this cell.")
    else:
        raise

## Your turn (exercises)

1. **Add a fourth agent.** Insert a "Senior Fact-Checker" task between the researcher and the analyst that flags any claim the brief can't support; feed it forward with `context`.
2. **Structured handoff.** Give `analysis_task` an `output_pydantic` model so the writer receives typed, predictable analysis instead of prose.
3. **Swap the process.** Try `Process.hierarchical` (with a manager LLM) and observe how task delegation changes.
4. **Prove memory works.** With `memory=True`, run the crew on a topic, then run it again asking it to "build on what you found last time" and check whether Run 2 references Run 1.
5. **Compare to Lab 03A.** You built memory by hand there and got it from the framework here. Which was clearer? Which would you reach for in production, and why?

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.

## Exercises

Code for exercises 1-5 follows. Design rationale for all five is in EXERCISE_NOTES.md.

Rationale, expected results, and the written answer to Exercise 5 are in **`EXERCISE_NOTES.md`**,
kept out of this notebook to leave the lab text unchanged.

In [ ]:
# =============================================================================
# EXERCISE SETUP -- helpers shared by exercises 1, 2, 3 and 4
# =============================================================================
import os
import re

OUT_ROOT = "outputs"


def save_output(relpath: str, text: str) -> str:
    """Write a task output under outputs/. On Colab this is /content/outputs."""
    path = os.path.join(OUT_ROOT, relpath)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as fh:
        fh.write(text)
    return path


def make_research_task(topic: str = TOPIC) -> Task:
    """A FRESH research task, worded exactly like the original research_task above.

    Why a factory and not the original object: a Task stores its result on `.output`, so
    passing the original to a new Crew would overwrite the baseline result saved in
    outputs/before-memory/ -- the control every exercise below compares against.

    Why the wording is copied verbatim rather than improved: the added fact-checker must be
    the only variable that changes.
    """
    return Task(
        description=f"""# Context
You are the first step; the analyst and writer build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{topic}""",
        expected_output="A structured research brief with clear section headers.",
        agent=researcher,
    )


async def run_crew(crew, label: str = "the crew") -> bool:
    """kickoff_async() with the same 503 guard the lab uses. True if the run finished."""
    try:
        await crew.kickoff_async()
        return True
    except Exception as e:
        if "503" in str(e):
            print(f"Gemini overloaded (503); {label} did not finish. Re-run in a few minutes.")
            return False
        raise


def task_text(task):
    """A task's raw output, or None if it never ran."""
    out = getattr(task, "output", None)
    return getattr(out, "raw", None) if out else None


print("Setup ready.")

In [ ]:
# =============================================================================
# EXERCISE 1 -- add a fourth agent: a Senior Fact-Checker between researcher and analyst
# =============================================================================
# Wiring (the context= lines ARE the change):
#   ex1_research_task   context=[]
#   ex1_factcheck_task  context=[research]                <-- NEW
#   ex1_analysis_task   context=[research, factcheck]
#   ex1_writing_task    context=[factcheck, analysis]     <-- note: NOT the raw brief
#
# The writer deliberately does not get the brief. The analysis already carries its substance,
# and withholding it stops the writer reaching past the audit to the unqualified original.

fact_checker = Agent(
    role="Senior Fact-Checker",
    goal="Flag every claim in the brief that the brief itself cannot support.",
    backstory=(
        "You audit evidence for a living and you are adversarial about it. You never soften a "
        "verdict to be agreeable, and you never add facts of your own."
    ),
    llm=llm,
    verbose=False,
)

ex1_research_task = make_research_task()

ex1_factcheck_task = Task(
    description="""# Context
The researcher's brief is available to you as context. It is the ONLY evidence you have.

# Task
Audit every substantive claim in the brief -- every figure, benchmark result, named system,
and causal assertion.

# Constraints
- One verdict per claim: SUPPORTED, UNSUPPORTED, OVERSTATED, or NEEDS-ATTRIBUTION.
- Quote each claim verbatim (<= 15 words) so later agents can match it by string.
- Judge the claim AS WRITTEN. A figure without the source that gives it meaning is
  NEEDS-ATTRIBUTION. A range presented as a floor ("over 20-30%") is OVERSTATED.
- Do NOT invent replacement facts or supply the missing source. Say what is missing.

# Format
A markdown table: Claim | Verdict | Why.
Then a section headed exactly "## Do not repeat downstream" listing, as `- ` bullets, every
claim that must be dropped or qualified, with the qualification it needs.""",
    expected_output=(
        "A markdown claim-audit table (Claim | Verdict | Why) followed by a "
        "'## Do not repeat downstream' bullet list."
    ),
    agent=fact_checker,
    context=[ex1_research_task],
)

ex1_analysis_task = Task(
    description="""# Context
Two documents are available to you: the researcher's brief, and the fact-checker's audit of it.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.
- The audit OVERRIDES the brief. Where they disagree, the audit wins.
- Never restate a claim from the audit's "Do not repeat downstream" list unqualified.
- If you cite a figure, carry its source in the same sentence.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets),
then a section headed exactly "## Claims dropped" naming what you left out and why.""",
    expected_output="A ranked analysis with a summary, implications, recommendations, and '## Claims dropped'.",
    agent=analyst,
    context=[ex1_research_task, ex1_factcheck_task],
)

ex1_writing_task = Task(
    description="""# Context
The fact-checker's audit AND the analysis are available to you. You do NOT have the original
brief -- by design.

# Task
Write a polished, publication-ready short report for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.
- If you use a figure, keep it exact and name its source in the same sentence.
- Never replace a specific figure with a vague quantifier ("nearly a third", "the majority").
- Anything on the audit's "Do not repeat downstream" list is off limits unqualified.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words.",
    agent=writer,
    context=[ex1_factcheck_task, ex1_analysis_task],
)

ex1_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[ex1_research_task, ex1_factcheck_task, ex1_analysis_task, ex1_writing_task],
    process=Process.sequential,
    verbose=False,
)

if await run_crew(ex1_crew, "the four-agent crew"):
    for header, task, fname in [
        ("RESEARCH BRIEF", ex1_research_task, "research_brief.md"),
        ("FACT-CHECK AUDIT (new agent)", ex1_factcheck_task, "factcheck_audit.md"),
        ("ANALYSIS (brief + audit)", ex1_analysis_task, "analysis.md"),
        ("FINAL REPORT (audit + analysis)", ex1_writing_task, "final_report.md"),
    ]:
        print(f"\n=== {header} ===")
        print(task.output.raw)
        save_output(f"after-factchecker/{fname}", task.output.raw)

In [ ]:
# =============================================================================
# EXERCISE 1 CHECK -- did the fact-checker actually catch the drift?
# =============================================================================
# Graded in plain Python, not by another LLM: an LLM auditing an LLM is how you get an
# agreeable rubber stamp. Same pattern as the deterministic checker in Lab 03A.
#
# The known drift in outputs/before-memory/ (see EXERCISE_NOTES.md for the full chain):
#   brief    "SWE-bench ... resolve over 20-30%"
#   analysis "resolve 20-30%"            <- SWE-bench attribution gone
#   report   "nearly a third"            <- figure gone

# Range form must come FIRST in the alternation, or the plain-number branch matches "20"
# out of "20-30%" and the range is lost.
FIGURE_RE = re.compile(
    r"\b\d+(?:\.\d+)?\s*(?:-|to|--|\u2013)\s*\d+(?:\.\d+)?\s*(?:%|percent)"
    r"|\b\d+(?:\.\d+)?\s*(?:%|percent)",
    re.I,
)

# Only fraction-style stand-ins count as drift. Including a bare "most" or "some" would fire on
# any hop that merely dropped a figure and happened to use the word -- a false failure.
VAGUE_RE = re.compile(
    r"\b(?:nearly|almost|about|roughly|around|over|under|more than|less than)?\s*"
    r"\b(?:a third|a quarter|a half|half|two[- ]thirds|three[- ]quarters|the majority|the bulk)\b",
    re.I,
)

VERDICTS = ("SUPPORTED", "UNSUPPORTED", "OVERSTATED", "NEEDS-ATTRIBUTION")


def norm_figure(s):
    """Normalize so '20-30%' and '20 to 30 percent' compare equal."""
    s = s.lower().replace("percent", "%")
    return re.sub(r"\s+", "", re.sub(r"\s*(?:--|\u2013|\bto\b)\s*", "-", s))


def figures(text):
    return {norm_figure(m.group()) for m in FIGURE_RE.finditer(text)}


def vague(text):
    return [m.group().strip() for m in VAGUE_RE.finditer(text)]


def drift_report(label, upstream, downstream, attributions=()):
    """A precise figure lost upstream + a vague stand-in gained downstream = drift."""
    up, down = figures(upstream), figures(downstream)
    f = {
        "figures_lost": sorted(up - down),
        "vague_downstream": vague(downstream),
        "attributions_lost": sorted(
            a for a in attributions
            if a.lower() in upstream.lower() and a.lower() not in downstream.lower()
        ),
    }
    f["substitution"] = bool(f["figures_lost"] and f["vague_downstream"])
    print(f"{label}")
    print(f"   figures lost: {f['figures_lost'] or '-'}    stand-ins added: {f['vague_downstream'] or '-'}")
    print(f"   attributions lost: {f['attributions_lost'] or '-'}    DRIFT: {'YES' if f['substitution'] else 'no'}")
    return f


# Baseline excerpts inlined so this runs on Colab, where outputs/before-memory/ is absent.
BASELINE_FALLBACK = {
    "research_brief.md": (
        "The industry standard for evaluating these agents is SWE-bench, a dataset of real-world "
        "GitHub issues. While early LLMs solved <2% of these issues, state-of-the-art agentic "
        "systems now resolve over 20-30% of these complex, multi-file problems autonomously."
    ),
    "analysis.md": (
        "While state-of-the-art agents can now resolve 20-30% of complex software engineering "
        "issues autonomously, their adoption is bottlenecked by inadequate testing infrastructure."
    ),
    "final_report.md": (
        "While cutting-edge agents can now solve nearly a third of complex engineering issues, "
        "their adoption is bottlenecked by a surprising obstacle: testing."
    ),
}


def read_baseline(fname):
    path = os.path.join(OUT_ROOT, "before-memory", fname)
    return open(path).read() if os.path.exists(path) else BASELINE_FALLBACK[fname]


# --- Probe 1: POSITIVE CONTROL. If the checker cannot find the drift we already know is in
# --- the baseline, its clean verdict on Exercise 1 would be meaningless. So it must fail loudly.
print("PROBE 1 -- positive control on the baseline run")
b_brief, b_analysis, b_report = (read_baseline(f) for f in
                                 ("research_brief.md", "analysis.md", "final_report.md"))
b1 = drift_report("  baseline hop 1 (brief -> analysis)", b_brief, b_analysis, ["SWE-bench"])
b2 = drift_report("  baseline hop 2 (analysis -> report)", b_analysis, b_report, ["SWE-bench"])
assert b2["substitution"] and b1["attributions_lost"] == ["SWE-bench"], \
    "Drift detector failed its positive control -- fix it before trusting anything below."
print("  positive control PASSES\n")

# --- Probes 2 and 3: grade the Exercise 1 run. ---
ex1 = [task_text(t) for t in (ex1_research_task, ex1_factcheck_task,
                              ex1_analysis_task, ex1_writing_task)]
if not all(ex1):
    print("Run the Exercise 1 cell first.")
else:
    brief, audit, analysis, report = ex1

    print("PROBE 2 -- did the same drift survive the four-agent pipeline?")
    h1 = drift_report("  ex1 hop 1 (brief -> analysis)", brief, analysis, ["SWE-bench"])
    h2 = drift_report("  ex1 hop 2 (analysis -> report)", analysis, report, ["SWE-bench"])

    print("\nPROBE 3 -- is the audit a real audit or a rubber stamp?")
    # Count verdicts per table row. Longest verdict first, so UNSUPPORTED is never
    # miscounted as SUPPORTED (it contains it as a substring).
    rows = [ln for ln in audit.splitlines() if ln.count("|") >= 2 and not set(ln) <= set("|-: ")]
    counts = {v: 0 for v in VERDICTS}
    for row in rows:
        for v in sorted(VERDICTS, key=len, reverse=True):
            if v in row.upper():
                counts[v] += 1
                break
    flagged = sum(counts.values()) - counts["SUPPORTED"]
    print(f"  verdicts: {counts}")
    print(f"  claims flagged (not SUPPORTED): {flagged}")
    has_dnr = "do not repeat downstream" in audit.lower()

    print("\nVERDICT")
    results = {
        "audit flagged >=1 claim (not a rubber stamp)": flagged > 0,
        "audit produced the '## Do not repeat downstream' handoff": has_dnr,
        "no figure-to-vague substitution at the analyst hop": not h1["substitution"],
        "no figure-to-vague substitution at the writer hop": not h2["substitution"],
        "SWE-bench attribution survived to the analysis": not h1["attributions_lost"],
    }
    for name, ok in results.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    # A FAIL here is a legitimate result worth reporting, not a broken cell.
    # See EXERCISE_NOTES.md, "Reading the Exercise 1 result".

In [ ]:
# =============================================================================
# EXERCISE 2 -- structured handoff: give analysis_task an output_pydantic model
# =============================================================================
from typing import List, Literal

from pydantic import BaseModel, Field


class Implication(BaseModel):
    """`rank` is what makes "rank by impact" checkable in code instead of a hope."""

    rank: int = Field(description="1 = highest impact. Contiguous, no ties, no gaps.")
    claim: str = Field(description="The implication, one sentence.")
    reasoning: str = Field(description="Why it follows from the research.")
    impact: Literal["high", "medium", "low"]
    # A Literal forces a value a consumer can branch on; a free-text field would
    # let the model answer "probably true".
    confidence: Literal["high", "medium", "low"] = Field(
        description="How well the audited research supports this. LOW if it rests on a flagged claim."
    )


class Recommendation(BaseModel):
    action: str = Field(description="A concrete action, phrased as an imperative.")
    rationale: str = Field(description="The implication it answers and why it is worth doing.")


class AnalysisOutput(BaseModel):
    """The typed contract between analyst and writer.

    dropped_claims exists so Exercise 1's audit trail survives the schema. A schema decides
    what CAN be said, so with no field for it the constrained output would silently delete
    the safety feature the fact-checker added.
    """

    executive_summary: str = Field(description="Exactly two sentences.")
    implications: List[Implication] = Field(description="Three, ranked, most impactful first.")
    recommendations: List[Recommendation] = Field(description="Two.")
    dropped_claims: List[str] = Field(
        default_factory=list,
        description="Claims from the audit's 'do not repeat downstream' list you left out, with why.",
    )


print("Schema fields the writer will receive:", list(AnalysisOutput.model_fields))

In [ ]:
# =============================================================================
# EXERCISE 2 (cont.) -- the typed crew. Builds on Exercise 1's four-agent shape.
# =============================================================================
# Two things that break QUIETLY if skipped:
#   1. expected_output must describe the JSON, not "a ranked analysis". CrewAI puts both the
#      schema and expected_output in the prompt; a prose-shaped one pulls against the schema.
#   2. The writer now receives serialized JSON, not paragraphs. Its description must say so and
#      name the fields, or it starts echoing "executive_summary:" into the report.

ex2_research_task = make_research_task()

ex2_factcheck_task = Task(
    description=ex1_factcheck_task.description,   # unchanged; the auditor is not the variable here
    expected_output=ex1_factcheck_task.expected_output,
    agent=fact_checker,
    context=[ex2_research_task],
)

ex2_analysis_task = Task(
    description="""# Context
Two documents are available to you: the researcher's brief, and the fact-checker's audit of it.

# Task
Analyze the research and produce ranked implications and recommendations.

# Constraints
- The audit OVERRIDES the brief. Where they disagree, the audit wins.
- Rank 1 is the most impactful. Ranks must be 1, 2, 3 -- no ties, no gaps.
- Set `confidence` to "low" for any implication resting on a claim the audit flagged.
- Put every claim you deliberately left out into `dropped_claims`, with a few words on why.

# Format
Return ONLY a JSON object matching the required schema. No markdown fence, no prose around it.""",
    expected_output=(
        "A JSON object with keys: executive_summary (string, exactly two sentences); "
        "implications (array of exactly 3 objects with rank:int, claim:string, reasoning:string, "
        "impact:'high'|'medium'|'low', confidence:'high'|'medium'|'low'); recommendations (array "
        "of exactly 2 objects with action:string, rationale:string); dropped_claims (array of strings)."
    ),
    agent=analyst,
    context=[ex2_research_task, ex2_factcheck_task],
    output_pydantic=AnalysisOutput,        # <-- the exercise
)

ex2_writing_task = Task(
    description="""# Context
Two documents are available to you: the fact-checker's audit (markdown), and the analysis --
which arrives as a JSON OBJECT, not prose. Its fields:

- `executive_summary` -- two sentences framing the picture
- `implications` -- three objects with `rank` (1 = most important), `claim`, `reasoning`,
  `impact`, `confidence`
- `recommendations` -- two objects with `action` and `rationale`
- `dropped_claims` -- claims the analyst excluded; these are OFF LIMITS to you

# Task
Write a polished, publication-ready short report for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.
- Lead with the rank-1 implication; rank order is the analyst's judgment of importance.
- Carry a `confidence` of "low" into the prose as a hedge; do not drop it.
- Say nothing that appears in `dropped_claims`.
- Never print a field name. You are writing prose FROM structured data, not transcribing it.
- Keep any figure exact and name its source in the same sentence.

# Format
A title, then 2-3 short paragraphs.""",
    expected_output="A short titled report of 200-300 words, with no field names visible.",
    agent=writer,
    context=[ex2_factcheck_task, ex2_analysis_task],
)

ex2_crew = Crew(
    agents=[researcher, fact_checker, analyst, writer],
    tasks=[ex2_research_task, ex2_factcheck_task, ex2_analysis_task, ex2_writing_task],
    process=Process.sequential,
    verbose=False,
)

ex2_ok = await run_crew(ex2_crew, "the typed-handoff crew")
if ex2_ok:
    # The payoff: an OBJECT, addressed by field. No parsing, no regex.
    analysis = ex2_analysis_task.output.pydantic
    if analysis is None:
        # Version-dependent: some CrewAI/LiteLLM combos fill .json_dict but not .pydantic.
        analysis = AnalysisOutput(**ex2_analysis_task.output.json_dict)

    print(f"=== TYPED ANALYSIS (type: {type(analysis).__name__}) ===")
    print(f"{analysis.executive_summary}\n")
    for imp in analysis.implications:
        print(f"  #{imp.rank} [impact={imp.impact}, confidence={imp.confidence}] {imp.claim}")
    for rec in analysis.recommendations:
        print(f"  -> {rec.action}")
    print(f"  dropped_claims: {analysis.dropped_claims or '(none)'}")

    print("\n=== FINAL REPORT (writer, from typed input) ===")
    print(ex2_writing_task.output.raw)

    save_output("typed-handoff/analysis.json", ex2_analysis_task.output.raw)
    save_output("typed-handoff/final_report.md", ex2_writing_task.output.raw)

In [ ]:
# =============================================================================
# EXERCISE 2 CHECK -- assertions that are IMPOSSIBLE against the prose version
# =============================================================================
# Try any of these against outputs/before-memory/analysis.md: there is no rank to compare,
# no enum to validate, and no list of dropped claims. You would be writing a parser and
# guessing. That is the argument for output_pydantic, stated as code rather than as a claim.

if not ex2_ok or task_text(ex2_writing_task) is None:
    print("Run the Exercise 2 crew cell first.")
else:
    analysis = ex2_analysis_task.output.pydantic or AnalysisOutput(**ex2_analysis_task.output.json_dict)
    report = ex2_writing_task.output.raw
    ranks = [i.rank for i in analysis.implications]
    words = len(report.split())

    # Field names must not leak into prose -- reports containing "rank:" are the classic symptom.
    leaked = [f for f in AnalysisOutput.model_fields if f in report] + \
             [f for f in ("rank", "confidence", "rationale") if re.search(rf"\b{f}\s*[:=]", report)]

    # Dropped claims must stay dropped. Match on distinctive words, since the writer
    # would paraphrase rather than quote.
    still_present = [
        c for c in analysis.dropped_claims
        if (kw := re.findall(r"[a-zA-Z]{6,}", c)[:3]) and all(w.lower() in report.lower() for w in kw)
    ]

    checks = {
        "ranks are a contiguous 1..N (no ties, no gaps)": sorted(ranks) == list(range(1, len(ranks) + 1)),
        "exactly 3 implications": len(analysis.implications) == 3,
        "exactly 2 recommendations": len(analysis.recommendations) == 2,
        "executive_summary is exactly 2 sentences":
            len([s for s in re.split(r"(?<=[.!?])\s+", analysis.executive_summary.strip()) if s]) == 2,
        "no schema field names leaked into the report": not leaked,
        "claims the analyst dropped do not reappear": not still_present,
        f"report is 200-300 words (got {words})": 200 <= words <= 300,
    }
    for name, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    if leaked:
        print(f"  leaked: {sorted(set(leaked))}")
    if still_present:
        print(f"  dropped claims that came back: {still_present}")

In [ ]:
# =============================================================================
# EXERCISE 3 -- Process.hierarchical, priced against Process.sequential
# =============================================================================
# The trap: if the tasks keep their explicit context=[...] wiring, the manager has nothing left
# to decide and a hierarchical run is a sequential run with a bigger token bill. So BOTH arms
# below use the SAME three context-free task descriptions, and differ only in who does the
# wiring:
#
#   sequential   -- agent= hard-wired by us; CrewAI passes each task the previous task's output
#   hierarchical -- agent= left unset; a manager agent (built from manager_llm) chooses the
#                   agent for each task and is the only thing routing work between them
#
# Why ONE run per arm supports a conclusion here: the load-bearing number is STRUCTURAL, not
# sampled. Hierarchical mode inserts manager turns whatever the model happens to say, so the
# call count and the token bill are properties of the topology -- they do not average out over
# reruns. Contrast Exercise 4, where the measured quantity was semantic recall and one sample
# proved little. Wall clock and specifics-propagation below ARE sampled: reported, not verdicted.
import time

# Config traps, all three of which raise or silently no-op if you get them wrong:
#   - manager_llm OR manager_agent, never both.
#   - the manager must NOT appear in agents= (a manager listed as a worker can assign to itself).
#   - Process.hierarchical with no manager at all is a validation error.


def make_unwired_tasks(assign: bool):
    """The same three tasks for both arms. assign=True hard-wires agent=; False leaves it open.

    No context= anywhere -- that is the whole point. The baseline crew's context=[research,
    analysis] is what a manager would otherwise have nothing to add to.

    The "carry the specifics through by name" constraint is in both arms identically, so the
    propagation measure in the next cell reflects the topology rather than one arm being asked
    to be more careful than the other.
    """
    research = Task(
        description=f"""# Context
You are the first step; other members of this crew build on your brief.

# Task
Produce a concise research brief on the topic in the Input section.

# Constraints
- Cover key facts, main challenges, and 2-3 notable examples.
- Be specific; cite concepts, not URLs.

# Format
Short sections with clear headers.

# Input (topic)
{TOPIC}""",
        expected_output="A structured research brief with clear section headers.",
        **({"agent": researcher} if assign else {}),
    )
    analysis = Task(
        description="""# Context
Another member of this crew has already produced a research brief on this topic. Work from that
brief; do not start a fresh review of the field.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.
- Carry the brief's specific systems, examples and figures through BY NAME AND NUMBER; do not
  generalise them into "many tools" or "a significant share".

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
        expected_output="A ranked analysis with a summary, implications, and recommendations.",
        **({"agent": analyst} if assign else {}),
    )
    writing = Task(
        description="""# Context
This crew has produced a research brief and an analysis of it.

# Task
Write a polished, publication-ready short report combining them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.
- Name the specific systems, examples and figures the brief established; do not round them off.

# Format
A title, then 2-3 short paragraphs.""",
        expected_output="A short titled report of 200-300 words.",
        **({"agent": writer} if assign else {}),
    )
    return [research, analysis, writing]


def usage_dict(crew):
    """crew.usage_metrics, tolerant of version drift (pydantic object, dict, or absent)."""
    m = getattr(crew, "usage_metrics", None)
    if m is None:
        return {}
    if isinstance(m, dict):
        return dict(m)
    keys = ("total_tokens", "prompt_tokens", "completion_tokens",
            "cached_prompt_tokens", "successful_requests")
    return {k: getattr(m, k) for k in keys if getattr(m, k, None) is not None}


def route_of(task):
    """Which agent ACTUALLY executed a task -- CrewAI records the role on the TaskOutput."""
    out = getattr(task, "output", None)
    if not out:
        return "(did not run)"
    return getattr(out, "agent", None) or "(not recorded)"


hier_results = {}


async def run_topology(arm: str, hierarchical: bool):
    tasks = make_unwired_tasks(assign=not hierarchical)
    kwargs = dict(agents=[researcher, analyst, writer], tasks=tasks, verbose=False)
    if hierarchical:
        kwargs.update(process=Process.hierarchical, manager_llm=llm)
    else:
        kwargs.update(process=Process.sequential)
    crew = Crew(**kwargs)

    print(f"\n--- ARM: {arm} ---")
    t0 = time.perf_counter()
    ok = await run_crew(crew, arm)
    elapsed = time.perf_counter() - t0
    if not ok:
        return

    outs = [task_text(t) for t in tasks]
    use = usage_dict(crew)
    hier_results[arm] = {"seconds": elapsed, "usage": use,
                         "routing": [route_of(t) for t in tasks], "outputs": outs}
    for name, text in zip(("research", "analysis", "report"), outs):
        if text:
            save_output(f"exercise3-{arm}/{name}.md", text)
    print(f"  {elapsed:.0f}s, {use.get('total_tokens', '?')} tokens, "
          f"{use.get('successful_requests', '?')} LLM calls -> {OUT_ROOT}/exercise3-{arm}/")


# Sequential first: it is the cheaper arm, so the baseline exists even if the hierarchical arm
# runs into a 503 partway through.
await run_topology("sequential", hierarchical=False)
await run_topology("hierarchical", hierarchical=True)

print("\nArms completed:", sorted(hier_results))

In [ ]:
# =============================================================================
# EXERCISE 3 CHECK -- what the manager's autonomy bought, and what it cost
# =============================================================================
# Run the EXERCISE 1 CHECK cell first -- specifics() reuses figures() from it.
#
# Exercise 4's check cell defines the same measure under the name entities(). It is repeated
# here rather than shared because that cell runs AFTER this one; a local definition means this
# cell stands on its own whatever order you execute in.

NAME_RE = re.compile(r"\b[A-Z][A-Za-z0-9]*(?:[-‑][A-Za-z0-9]+)*\b")

# Section headers our own prompts dictate -- they appear in every arm regardless of handoff
# quality, so counting them would inflate propagation identically on both sides and hide the
# difference we are looking for.
NAME_STOP = {"Context", "Task", "Constraints", "Format", "Input", "Executive", "Summary",
             "Key", "Implications", "Recommendations", "Title", "This", "The", "Another"}


def specifics(text):
    """Named systems and figures -- the details a paraphrase drops and a real handoff keeps."""
    found = set()
    for m in NAME_RE.finditer(text or ""):
        tok = m.group()
        if len(tok) < 4 or tok in NAME_STOP:
            continue
        # An internal capital, hyphen or digit is what separates a real name from any word that
        # merely began a sentence.
        if not (re.search(r"[A-Z0-9]", tok[1:]) or "-" in tok or "‑" in tok):
            continue
        found.add(tok.lower())
    return found | figures(text or "")


HAND_WIRING = ["Senior Research Analyst", "Senior Strategy Analyst", "Senior Technical Writer"]


def _n(v):
    return "n/a" if v is None else f"{v:,}"


if len(hier_results) < 2:
    print("Both arms have not completed -- re-run the cell above (a 503 may have cut one short).")
else:
    seq, hier = hier_results["sequential"], hier_results["hierarchical"]

    rows = [(k, seq["usage"].get(k), hier["usage"].get(k))
            for k in ("total_tokens", "prompt_tokens", "completion_tokens", "successful_requests")]
    rows.append(("seconds", round(seq["seconds"]), round(hier["seconds"])))

    print(f"{'metric':<22}{'sequential':>13}{'hierarchical':>15}{'ratio':>9}")
    for k, a, b in rows:
        ratio = f"{b / a:.2f}x" if isinstance(a, (int, float)) and isinstance(b, (int, float)) and a else "--"
        print(f"{k:<22}{_n(a):>13}{_n(b):>15}{ratio:>9}")
    print("(tokens and calls are structural; seconds is a single noisy sample)")

    print("\nROUTING -- which agent actually executed each task:")
    for i, name in enumerate(("research", "analysis", "report")):
        h = hier["routing"][i]
        verdict = "== hand-wiring" if h == HAND_WIRING[i] else f"!= hand-wired {HAND_WIRING[i]}"
        print(f"  {name:<9} sequential: {seq['routing'][i]:<26} hierarchical: {h:<26} {verdict}")
    matched = sum(hier["routing"][i] == HAND_WIRING[i] for i in range(3))

    print("\nPROPAGATION -- specifics from the brief that survive into the final report:")
    for arm, d in (("sequential", seq), ("hierarchical", hier)):
        brief, report = specifics(d["outputs"][0]), specifics(d["outputs"][2])
        kept = len(brief & report)
        pct = f"{kept / len(brief):.0%}" if brief else "n/a"
        print(f"  {arm:<14}{kept:>4} of {len(brief):<4} kept  ({pct})")
    missing = [n for n, t in zip(("research", "analysis", "report"), hier["outputs"]) if not t]
    if missing:
        print(f"  WARNING: the hierarchical arm produced no output for: {', '.join(missing)}")

    a, b = seq["usage"].get("total_tokens"), hier["usage"].get("total_tokens")
    if a and b:
        print(f"\nPRICE OF AUTONOMY: the manager cost {b / a:.2f}x the tokens of sequential "
              f"for the same three tasks.")
    ca, cb = seq["usage"].get("successful_requests"), hier["usage"].get("successful_requests")
    if ca and cb:
        print(f"MANAGER OVERHEAD: {cb - ca} extra LLM calls ({cb} vs {ca}) -- the manager's own "
              f"turns, one or more per task.")

    print(f"ROUTING FIDELITY: {matched}/3 tasks went to the agent we would have hand-wired.")
    if matched == 3:
        print("^ the manager rediscovered the hand-wiring exactly. On a pipeline this obvious the")
        print("  extra tokens bought a decision that was already known -- which is the finding:")
        print("  hierarchical mode is worth its bill only when the routing is NOT known up front")
        print("  (unknown task count, agent chosen by intermediate results, retry on bad work).")
    elif matched == 0:
        print("^ the manager routed every task elsewhere. Read outputs/exercise3-hierarchical/")
        print("  before calling that wrong -- but a report written by the researcher is a smell.")
    else:
        print("^ partial agreement. The disagreeing task is the interesting one: compare the two")
        print("  arms' outputs for it and decide whether the manager's choice was better.")

In [ ]:
# =============================================================================
# EXERCISE 4 -- prove memory works, against a memory=False control
# =============================================================================
# Why the optional memory_crew cell above does NOT prove it: it runs the same crew, on the same
# topic, with the SAME task descriptions, twice. Identical prompts to a temperature-0.3 model
# already produce similar output, so every bit of the resemblance is explained without memory.
#
# This version changes two things:
#   1. Run 2's prompt is DIFFERENT -- it asks for specifics only memory could supply, and offers
#      an explicit "NO PRIOR CONTEXT RECALLED" escape so a blank is distinguishable from a guess.
#   2. There is a CONTROL arm: same prompts, memory=False. On a shared topic most of the overlap
#      between two runs is topic, not memory. The control measures that floor; only the
#      DIFFERENCE between arms is evidence.
import shutil
import time

# Must be set BEFORE any memory-backed Crew is constructed -- CrewAI resolves the storage path
# when it builds its memory objects. If you already ran the memory_crew cell above in this
# session, restart the runtime so this takes effect.
STORAGE_DIR = os.path.abspath("crewai_lab03b_storage")
os.environ["CREWAI_STORAGE_DIR"] = STORAGE_DIR

MEM_TOPIC = TOPIC   # same topic as the baseline, so the control's overlap floor is realistic

EMBEDDER = {
    "provider": "google-generativeai",
    "config": {"api_key": api_key, "model_name": "gemini-embedding-001"},
}


def make_run1_tasks():
    """Run 1: an ordinary run. Identical wording in both arms."""
    research = make_research_task(MEM_TOPIC)
    analysis = Task(
        description="""# Context
The researcher's brief is available to you as context.

# Task
Analyze the research and draw the most important implications and recommendations.

# Constraints
- Rank implications by impact; back each claim with reasoning.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
        expected_output="A ranked analysis with a summary, implications, and recommendations.",
        agent=analyst,
        context=[research],
    )
    writing = Task(
        description="""# Context
The research brief AND the analysis are available to you as context.

# Task
Write a polished, publication-ready short report combining them for a general audience.

# Constraints
- 200-300 words; clear and engaging; no jargon dumps.

# Format
A title, then 2-3 short paragraphs.""",
        expected_output="A short titled report of 200-300 words.",
        agent=writer,
        context=[research, analysis],
    )
    return [research, analysis, writing]


def make_run2_tasks():
    """Run 2: asks for something ONLY memory can supply.

    The "## What I established last time" section is the measurement surface. With working
    memory it names Run 1's actual systems and figures; without it the crew must either take
    the offered blank (honest) or invent specifics (confabulation) -- and the entity-overlap
    check in the next cell tells those two apart.
    """
    research = Task(
        description=f"""# Context
You have researched this topic before, in an earlier session. Build on what you found last
time rather than starting over.

# Task
Extend your earlier research on the topic in the Input section.

# Constraints
- Begin with a section headed exactly "## What I established last time" listing the SPECIFIC
  systems, benchmarks, examples and figures you covered previously -- by name and by number.
- If you genuinely cannot recall anything specific from an earlier session, say exactly
  "NO PRIOR CONTEXT RECALLED" under that heading. Do NOT guess or reconstruct what you
  probably said -- an invented recollection is worse than an admitted blank.
- Then cover what you did NOT cover last time. Do not repeat the earlier ground.

# Format
The recall section, then short sections with clear headers.

# Input (topic)
{MEM_TOPIC}""",
        expected_output="A recall section, then new research extending the earlier brief.",
        agent=researcher,
    )
    analysis = Task(
        description="""# Context
The researcher's extended brief is available to you as context.

# Task
Analyze what is NEW relative to the earlier session and draw the implications.

# Constraints
- Rank implications by impact; back each claim with reasoning.
- Note explicitly which implications are new since last time.

# Format
Executive Summary (2 sentences), Key Implications (3 bullets), Recommendations (2 bullets).""",
        expected_output="A ranked analysis of what is new.",
        agent=analyst,
        context=[research],
    )
    writing = Task(
        description="""# Context
The extended brief AND the analysis are available to you as context.

# Task
Write a short follow-up report that builds on the earlier one.

# Constraints
- 200-300 words.
- Reference the earlier findings by name where you build on them.

# Format
A title, then 2-3 short paragraphs.""",
        expected_output="A short titled follow-up report of 200-300 words.",
        agent=writer,
        context=[research, analysis],
    )
    return [research, analysis, writing]


mem_results = {}


async def run_arm(arm: str, use_memory: bool):
    """Run 1 then Run 2 through ONE crew instance.

    Mutating crew.tasks instead of building a second crew matters: a Crew builds its memory
    objects at construction, and in some CrewAI versions short-term memory is scoped to that
    instance. Reusing the instance gives memory its best possible shot. The control performs the
    identical mutation, so the two arms differ in exactly one flag.
    """
    print(f"\n--- ARM: {arm} (memory={use_memory}) ---")
    run1 = make_run1_tasks()
    kwargs = dict(agents=[researcher, analyst, writer], process=Process.sequential, verbose=False)
    if use_memory:
        kwargs.update(memory=True, embedder=EMBEDDER)
    crew = Crew(tasks=run1, **kwargs)

    print("  Run 1 ...")
    if not await run_crew(crew, f"{arm} run 1"):
        return
    run1_brief = run1[0].output.raw

    run2 = make_run2_tasks()
    crew.tasks = run2
    print("  Run 2 (asked to build on last time) ...")
    if not await run_crew(crew, f"{arm} run 2"):
        return

    mem_results[arm] = {"run1": run1_brief, "run2": run2[0].output.raw}
    for name, text in [("run1_brief", run1_brief), ("run2_brief", run2[0].output.raw),
                       ("run2_report", run2[2].output.raw)]:
        save_output(f"exercise4-{arm}/{name}.md", text)
    print(f"  done -> {OUT_ROOT}/exercise4-{arm}/")


# CONTROL FIRST, while no memory store exists at all -- so it cannot be contaminated even if
# the wipe below were to fail.
await run_arm("control-no-memory", use_memory=False)

def wipe_storage(path):
    """Clear the memory store, tolerating live ChromaDB file handles.

    A plain rmtree raises OSError 39 (Directory not empty) when a Chroma client from the
    memory_crew cell above is still open: files reappear between rmtree's scandir and its final
    rmdir. Three tolerant passes, then move the directory aside -- rename succeeds even on open
    files, so the memory arm starts from an empty path either way. The stale copy is left on disk
    for inspection rather than force-deleted.
    """
    for _ in range(3):
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.exists(path):
            return "wiped"
        time.sleep(0.5)                 # give Chroma a moment to finish flushing
    stale = f"{path}.stale"
    shutil.rmtree(stale, ignore_errors=True)
    os.rename(path, stale)
    return f"still held open -- moved aside to {os.path.basename(stale)}"


if os.path.isdir(STORAGE_DIR):
    print(f"\n{STORAGE_DIR}: {wipe_storage(STORAGE_DIR)}")
await run_arm("with-memory", use_memory=True)

print("\nArms completed:", sorted(mem_results))

In [ ]:
# =============================================================================
# EXERCISE 4 CHECK -- measure recall: entity overlap, memory arm vs control
# =============================================================================
# Run the EXERCISE 1 CHECK cell first -- entities() reuses figures() from it.
#
# An "entity" is the kind of specific a paraphrase does NOT preserve: a named system
# (internal capital, hyphen, or digit) or a figure. Plain lowercase prose words are excluded
# on purpose -- two documents on one topic share those regardless of memory.

STOPWORDS = {
    "software", "development", "developer", "developers", "agent", "agents", "workflow",
    "workflows", "code", "coding", "impact", "context", "session", "research", "brief",
    "report", "analysis", "last", "time", "previously", "earlier", "established",
    "however", "furthermore", "moreover", "therefore", "because", "while", "these", "their",
}

ENTITY_RE = re.compile(r"\b[A-Z][A-Za-z0-9]*(?:[-\u2011][A-Za-z0-9]+)*\b")

BACKREF_RE = re.compile(
    r"\b(?:last time|previously|earlier session|prior session|as (?:I|we) (?:found|noted|established)|"
    r"in my earlier|building on|previous(?:ly)? (?:brief|research|report))\b", re.I)


def entities(text):
    found = set()
    for m in ENTITY_RE.finditer(text):
        tok = m.group()
        if len(tok) < 4 or tok.lower() in STOPWORDS:
            continue
        # Require an internal capital, hyphen or digit -- the marks of a real name. Without this,
        # any word that merely started a sentence would count.
        if not (re.search(r"[A-Z0-9]", tok[1:]) or "-" in tok or "\u2011" in tok):
            continue
        found.add(tok.lower())
    return found | figures(text)


def recall_section(text):
    """Just the '## What I established last time' block, if the model produced one."""
    m = re.search(r"##\s*What I established last time(.*?)(?=\n##\s|\Z)", text, re.S | re.I)
    return m.group(1).strip() if m else ""


if len(mem_results) < 2:
    print("Both arms have not completed -- re-run the cell above (a 503 may have cut one short).")
else:
    rows = {}
    for arm in ("control-no-memory", "with-memory"):
        r = mem_results[arm]
        e1, e2 = entities(r["run1"]), entities(r["run2"])
        sec = entities(recall_section(r["run2"])) if recall_section(r["run2"]) else set()
        rows[arm] = {
            "run1 entities": len(e1),
            "shared run1->run2": len(e1 & e2),
            "entity recall": (len(e1 & e2) / len(e1)) if e1 else 0.0,
            "recall-section: real": len(sec & e1),
            "recall-section: invented": len(sec - e1),
            # Near-worthless alone: the Run 2 prompt TELLS the crew it has been here before, so
            # both arms will say "previously" whether or not anything was recalled.
            "backref phrases": len(BACKREF_RE.findall(r["run2"])),
            "declared blank": "NO PRIOR CONTEXT RECALLED" in r["run2"].upper(),
        }

    print(f"{'metric':<28}{'control':>12}{'with-memory':>14}")
    for metric in rows["control-no-memory"]:
        c, m_ = rows["control-no-memory"][metric], rows["with-memory"][metric]
        fmt = (lambda v: f"{v:.0%}") if metric == "entity recall" else str
        print(f"{metric:<28}{fmt(c):>12}{fmt(m_):>14}")

    delta = rows["with-memory"]["entity recall"] - rows["control-no-memory"]["entity recall"]
    print(f"\nENTITY-RECALL DELTA (memory - control): {delta:+.0%}")
    print("^ this, not the raw recall number, is the evidence.")

    if delta >= 0.15:
        print("VERDICT: memory shows a real effect.")
    elif delta > 0.05:
        print("VERDICT: weak positive -- within one sample's variance. Re-run before claiming it.")
    else:
        print("VERDICT: no measurable memory effect on this run. This is a legitimate result;")
        print("see EXERCISE_NOTES.md for the likely causes and why NOT to tune the prompt.")

    ctrl = rows["control-no-memory"]
    if not ctrl["declared blank"] and ctrl["recall-section: invented"] > 0:
        print(f"\nNOTE: the control did not take the offered blank and named "
              f"{ctrl['recall-section: invented']} specifics absent from its own Run 1 --")
        print("told it had a history, it manufactured one.")

    print("\nOn disk:")
    if os.path.isdir(STORAGE_DIR):
        for root, _d, files in os.walk(STORAGE_DIR):
            for f in sorted(files):
                p = os.path.join(root, f)
                print(f"  {os.path.relpath(p, STORAGE_DIR):<54}{os.path.getsize(p):>10,} bytes")
    else:
        print(f"  {STORAGE_DIR} does not exist -- nothing persisted. If the memory arm ran,")
        print("  CREWAI_STORAGE_DIR was resolved before the cell set it: restart the runtime.")

In [ ]:
# =============================================================================
# EXERCISE 5 -- compare to Lab 03A
# =============================================================================
# This exercise asks for a written answer, not code. It is in EXERCISE_NOTES.md
# (section "Exercise 5"), to keep prose out of the notebook.
print("Exercise 5: written answer in EXERCISE_NOTES.md")

## Your turn (exercises)

1. **Add a fourth agent.** Insert a "Senior Fact-Checker" task between the researcher and the analyst that flags any claim the brief can't support; feed it forward with `context`.
2. **Structured handoff.** Give `analysis_task` an `output_pydantic` model so the writer receives typed, predictable analysis instead of prose.
3. **Swap the process.** Try `Process.hierarchical` (with a manager LLM) and observe how task delegation changes.
4. **Prove memory works.** With `memory=True`, run the crew on a topic, then run it again asking it to "build on what you found last time" and check whether Run 2 references Run 1.
5. **Compare to Lab 03A.** You built memory by hand there and got it from the framework here. Which was clearer? Which would you reach for in production, and why?

When you're done, save a copy (**File -> Save a copy in Drive**) and submit your notebook link via Canvas.